# Administration CSV generatorTurns a GeoJSON of administrative boundaries into the CSV that`administration_csv_seeder` imports.```GeoJSON -> this notebook -> storage/administrations/*.csv -> seeder -> workspace```Everything is driven by `config.json`. See `README.md` for the fieldreference; run the cells in order.

In [ ]:
import csvimport jsonimport refrom collections import Counter, defaultdictfrom pathlib import Pathdef find_repo_root(start=None):    """Walk up to the directory holding dc.sh.    Config paths are repo-root-relative so the same `output` value works    whether the notebook is launched from its own directory or from the    repo root.    """    current = (start or Path.cwd()).resolve()    for candidate in [current, *current.parents]:        if (candidate / "dc.sh").is_file():            return candidate    raise RuntimeError("could not locate the repo root (no dc.sh found)")ROOT = find_repo_root()HERE = ROOT / "scripts" / "administration_csv_generator"config_path = HERE / "config.json"if not config_path.is_file():    config_path = HERE / "config.json.example"    print(f"NOTE: no config.json; falling back to {config_path.name}.")    print("      Copy it to config.json and edit before generating.\n")CONFIG = json.loads(config_path.read_text())INPUT_GEOJSON = ROOT / CONFIG["input"]OUTPUT_CSV = ROOT / CONFIG["output"]OPTIONS = CONFIG.get("options") or {}NA_VALUES = {str(v) for v in OPTIONS.get("na_values", ["NA", ""])}SPLIT_CAMEL_CASE = bool(OPTIONS.get("split_camel_case", False))print("config:", config_path)print("input :", INPUT_GEOJSON, "(exists)" if INPUT_GEOJSON.is_file() else "(MISSING)")print("output:", OUTPUT_CSV)

## Step 1 - Preview the propertiesLoad the file and look at one feature. Every mapping decision is made fromthis list, so run this before editing `config.json`.

In [ ]:
with INPUT_GEOJSON.open() as handle:    geojson = json.load(handle)features = geojson.get("features", [])print(f"{len(features)} features\n")sample = features[0]["properties"]width = max(len(k) for k in sample)print("-- properties of feature[0] --")for key, value in sample.items():    print(f"  {key:<{width}} = {value!r}")

In [ ]:
# A property present on only some features cannot drive a level: the rows# missing it produce blank names, which the seeder rejects as a hole in# the path.present = Counter()for feature in features:    present.update(feature["properties"].keys())print("-- coverage --")for key, count in present.most_common():    flag = "" if count == len(features) else "   <-- PARTIAL"    print(f"  {key:<12} {count}/{len(features)}{flag}")# GADM writes the string "NA" rather than null.placeholder = Counter()for feature in features:    for key, value in feature["properties"].items():        if isinstance(value, str) and value.strip() in NA_VALUES:            placeholder[key] += 1if placeholder:    print("\n-- values matching na_values (treated as empty) --")    for key, count in placeholder.most_common():        print(f"  {key:<12} {count}/{len(features)}")

In [ ]:
def suggest_config(properties):    """A starting `labels` + `properties` block, guessed from the file.    GADM uses COUNTRY/GID_0 for the root and NAME_<n>/GID_<n> below it.    Labels are the one thing that cannot be guessed: only the deepest tier    carries ENGTYPE_<n>, so everything else falls back to a placeholder    you are expected to replace.    """    depths = sorted(        int(m.group(1))        for key in properties        for m in [re.fullmatch(r"NAME_(\d+)", key)]        if m    )    labels, props = {}, {}    if "COUNTRY" in properties:        labels["0"] = "National"        props["0_name"] = "COUNTRY"        if "GID_0" in properties:            props["0_code"] = "GID_0"    for depth in depths:        engtype = properties.get(f"ENGTYPE_{depth}")        labels[str(depth)] = (            engtype            if isinstance(engtype, str) and engtype.strip() not in NA_VALUES            else f"Level {depth}"        )        props[f"{depth}_name"] = f"NAME_{depth}"        if f"GID_{depth}" in properties:            props[f"{depth}_code"] = f"GID_{depth}"    return {"labels": labels, "properties": props}print("Suggested config -- paste into config.json and fix the labels:\n")print(json.dumps(suggest_config(sample), indent=2))

## Step 2 - Read the mapping`labels` decides what each tier is *called*. The seeder derives`Levels.name` from it, so `"1": "Province"` creates a level named"Province" that people see throughout the app — it is not just a columnheading.

In [ ]:
def parse_levels(config):    """config -> [{level, label, name_prop, code_prop}], ordered."""    props = config.get("properties") or {}    labels = {str(k): v for k, v in (config.get("labels") or {}).items()}    depths = sorted({        int(m.group(1))        for key in props        for m in [re.fullmatch(r"(\d+)_(name|code)", key)]        if m    })    levels = []    for depth in depths:        name_prop = props.get(f"{depth}_name")        if not name_prop:            raise ValueError(f"config.properties is missing '{depth}_name'")        label = labels.get(str(depth))        if not label:            label = f"Level {depth}"            print(                f"WARNING: no labels['{depth}']; using {label!r}. "                "The workspace will show that as the tier's name."            )        levels.append({            "level": depth,            "label": str(label).strip(),            "name_prop": name_prop,            "code_prop": props.get(f"{depth}_code"),        })    return levelsLEVELS = parse_levels(CONFIG)for entry in LEVELS:    code_prop = entry["code_prop"] or "-"    print(        f"  level {entry['level']}  {entry['label']:<16} "        f"name={entry['name_prop']:<10} code={code_prop}"    )

In [ ]:
_CAMEL = [    (re.compile(r"(?<=[a-z])(?=[A-Z])"), " "),      # AcehBarat -> Aceh Barat    (re.compile(r"(?<=[A-Z])(?=[A-Z][a-z])"), " "),  # DKIJakarta -> DKI Jakarta]def clean(value):    """Property value -> cell value. Empty string means 'no value'."""    if value is None:        return ""    text = str(value).strip()    if text in NA_VALUES:        return ""    if SPLIT_CAMEL_CASE:        for pattern, replacement in _CAMEL:            text = pattern.sub(replacement, text)    return textprint(f"split_camel_case = {SPLIT_CAMEL_CASE}\n")print("-- how names will be written --")for feature in features[:8]:    props = feature["properties"]    print("  " + " / ".join(        clean(props.get(e["name_prop"])) for e in LEVELS[1:]    ))

## Step 3 - ValidateEvery check here mirrors a rule the seeder enforces, so a clean run meansthe import will succeed.

In [ ]:
def validate(features, levels):    problems = []    depths = [e["level"] for e in levels]    if depths != list(range(len(depths))):        problems.append(f"levels must be contiguous from 0; got {depths}")    labels = [e["label"] for e in levels]    for label in labels:        if label.lower() == "code":            problems.append(                f"label {label!r} collides with the '<level>_Code' column"            )    if len(set(labels)) != len(labels):        problems.append(f"labels must be unique; got {labels}")    keys = set()    for feature in features:        keys.update(feature["properties"].keys())    for entry in levels:        for role in ("name_prop", "code_prop"):            prop = entry[role]            if prop and prop not in keys:                problems.append(                    f"level {entry['level']}: {role} {prop!r} is not in "                    "the file"                )    # A blank tier with a non-blank descendant is a hole in the path.    # seed_administrations walks parent -> child and cannot bridge one.    holes = 0    for index, feature in enumerate(features):        props = feature["properties"]        blank_at = None        for entry in levels:            value = clean(props.get(entry["name_prop"]))            if not value:                if blank_at is None:                    blank_at = entry                continue            if blank_at is not None:                holes += 1                if holes <= 3:                    problems.append(                        f"feature[{index}]: {blank_at['name_prop']} is "                        f"blank but {entry['name_prop']} is not -- a path "                        "cannot skip a tier"                    )                break    if holes > 3:        problems.append(f"...and {holes - 3} more features with holes")    return problemsissues = validate(features, LEVELS)if issues:    print("PROBLEMS -- fix config.json before continuing:\n")    for issue in issues:        print("  -", issue)else:    print("Mapping is valid.")

## Step 4 - Build the rowsOne row per deepest unit, deduplicated on the full path, so a GeoJSONcarrying several polygons for the same unit contributes one row.

In [ ]:
header = []for entry in LEVELS:    header.append(f"{entry['level']}_{entry['label']}")    if entry["code_prop"]:        header.append(f"{entry['level']}_Code")rows, seen = [], set()for feature in features:    props = feature["properties"]    row, path = [], []    for entry in LEVELS:        name = clean(props.get(entry["name_prop"]))        path.append(name)        row.append(name)        if entry["code_prop"]:            row.append(clean(props.get(entry["code_prop"])))    key = tuple(path)    if key in seen:        continue    seen.add(key)    rows.append(row)print("header:", ",".join(header))print(    f"\n{len(rows)} rows from {len(features)} features "    f"({len(features) - len(rows)} duplicate paths collapsed)\n")for row in rows[:5]:    print("  " + ",".join(row))

## Step 5 - Sanity checks

In [ ]:
name_index = {    entry["level"]: header.index(f"{entry['level']}_{entry['label']}")    for entry in LEVELS}print("-- distinct units per tier --")for entry in LEVELS:    column = f"{entry['level']}_{entry['label']}"    values = {r[name_index[entry["level"]]] for r in rows}    values.discard("")    print(f"  level {entry['level']} {column:<20} {len(values)}")# The seeder requires exactly one level-0 value: a workspace has one root# (unique_root_administration_per_tenant).roots = {r[name_index[LEVELS[0]["level"]]] for r in rows}print(f"\n-- root values: {sorted(roots)}")if len(roots) != 1:    print("  PROBLEM: the seeder requires exactly one.")# Why the seeder keys on (name, level, parent, tenant) rather than on name# alone: these would silently collapse under a name-only upsert.if len(LEVELS) >= 2:    leaf = LEVELS[-1]["level"]    parents = defaultdict(set)    for row in rows:        parents[row[name_index[leaf]]].add(            tuple(row[name_index[e["level"]]] for e in LEVELS[:-1])        )    shared = {n: p for n, p in parents.items() if len(p) > 1}    print(f"\n-- {len(shared)} leaf names appear under more than one parent")    for name, paths in sorted(shared.items(), key=lambda kv: -len(kv[1]))[:3]:        print(f"     {name!r} under {len(paths)} parents")# Case-insensitive sibling collisions DO merge: seed_administrations# matches on name__iexact within a parent, so "Kota Bogor" and "KOTA BOGOR"# under the same parent become one unit.merges = 0folded = defaultdict(set)for row in rows:    parent = None    for entry in LEVELS:        name = row[name_index[entry["level"]]]        if not name:            break        folded[(entry["level"], name.lower(), parent)].add(name)        parent = (entry["level"], name.lower(), parent)for key, variants in folded.items():    if len(variants) > 1:        merges += 1        if merges <= 3:            print(f"\n  MERGE: level {key[0]} {sorted(variants)} "                  "differ only by case and will become one unit")if merges:    print(f"\n  {merges} case-insensitive collisions total.")

## Step 6 - Write the CSV

In [ ]:
if issues:    raise SystemExit("Step 3 reported problems; fix config.json first.")OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as handle:    writer = csv.writer(handle)    writer.writerow(header)    writer.writerows(rows)print(f"wrote {len(rows)} rows to {OUTPUT_CSV} "      f"({OUTPUT_CSV.stat().st_size:,} bytes)")print("\n-- first 3 lines --")with OUTPUT_CSV.open() as handle:    for _ in range(3):        print("  " + handle.readline().rstrip())# The seeder takes a path relative to STORAGE_PATH, which is bind-mounted# from the repo's storage/ directory.try:    relative = OUTPUT_CSV.relative_to(ROOT / "storage")    print(f"\n--source {relative}")except ValueError:    print("\nNOTE: output is outside storage/, so the backend container "          "cannot read it. Point `output` at storage/administrations/.")

## Step 7 - Import into a workspaceDry-run first: it validates the whole file and rolls back.```bash./dc.sh exec backend python manage.py administration_csv_seeder \    --source administrations/indonesia.csv --tenant <subdomain> --dry-run./dc.sh exec backend python manage.py administration_csv_seeder \    --source administrations/indonesia.csv --tenant <subdomain>```If the workspace already has a root under a different name, the seederstops and names both rather than guessing; pass `--rename-root` to acceptthe file's value.